#### 准备数据

In [7]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    # 归一化：像素值从[0,255]缩放到[0,1]
    x = x / 255.0
    x_test = x_test / 255.0
    return (x, y), (x_test, y_test)

print('TensorFlow 版本:', tf.__version__)

TensorFlow 版本: 2.10.0


#### 定义卷积和池化函数

In [8]:
def conv2d(x, W):
    """
    二维卷积函数
    x: 输入特征图，shape = (batch, height, width, channels)
    W: 卷积核，shape = (kernel_h, kernel_w, in_channels, out_channels)
    每一维滑动步长都是1，padding方式选择same（输出尺寸和输入相同）
    """
    return tf.nn.conv2d(x, W, strides=[1, 1, 1, 1], padding='SAME')


def max_pool_2x2(x):
    """
    2x2最大池化函数
    x: 输入特征图
    池化窗口：2x2；滑动步长：2；padding方式：same
    作用：把特征图的高和宽各缩小一半
    """
    return tf.nn.max_pool2d(x, ksize=[2, 2], strides=[2, 2], padding='SAME')

#### 建立CNN模型

In [9]:
class myCNNModel:
    def __init__(self):
        # ===== 卷积层1的参数 =====
        # 空1: 卷积核权重，patch 7x7，输入通道1（灰度图），输出通道32
        self.W_conv1 = tf.Variable(tf.random.truncated_normal([7, 7, 1, 32], stddev=0.1))
        # 空2: 偏置，大小等于输出通道数32
        self.b_conv1 = tf.Variable(tf.constant(0.1, shape=[32]))

        # ===== 卷积层2的参数 =====
        # 空5: 卷积核权重，patch 5x5，输入通道32（上一层输出），输出通道64
        self.W_conv2 = tf.Variable(tf.random.truncated_normal([5, 5, 32, 64], stddev=0.1))
        # 空6: 偏置，大小等于输出通道数64
        self.b_conv2 = tf.Variable(tf.constant(0.1, shape=[64]))

        # ===== 全连接层1的参数 =====
        # 经过2次2x2池化后：28→14→7，特征图尺寸是7x7x64
        self.W_fc1 = tf.Variable(tf.random.truncated_normal([7 * 7 * 64, 1024], stddev=0.1))
        self.b_fc1 = tf.Variable(tf.constant(0.1, shape=[1024]))

        # ===== 全连接层2（输出层）的参数 =====
        self.W_fc2 = tf.Variable(tf.random.truncated_normal([1024, 10], stddev=0.1))
        self.b_fc2 = tf.Variable(tf.constant(0.1, shape=[10]))

    def __call__(self, x, training=False):
        # 把输入从(batch,28,28)变成(batch,28,28,1)，最后一维1表示灰度图1个通道
        x = tf.reshape(x, [-1, 28, 28, 1])

        # 空3: 卷积层1 = conv2d + ReLU激活
        # 输入:28x28x1 → 输出:28x28x32（same padding尺寸不变）
        h_conv1 = tf.nn.relu(conv2d(x, self.W_conv1) + self.b_conv1)

        # 空4: 池化层1 = 2x2最大池化
        # 输入:28x28x32 → 输出:14x14x32
        h_pool1 = max_pool_2x2(h_conv1)

        # 空7: 卷积层2 = conv2d + ReLU激活
        # 输入:14x14x32 → 输出:14x14x64（same padding尺寸不变）
        h_conv2 = tf.nn.relu(conv2d(h_pool1, self.W_conv2) + self.b_conv2)

        # 空8: 池化层2 = 2x2最大池化
        # 输入:14x14x64 → 输出:7x7x64
        h_pool2 = max_pool_2x2(h_conv2)

        # 把7x7x64的特征图压平成一维向量，输入全连接层
        h_pool2_flat = tf.reshape(h_pool2, [-1, 7 * 7 * 64])

        # 全连接层1 + ReLU
        h_fc1 = tf.nn.relu(tf.matmul(h_pool2_flat, self.W_fc1) + self.b_fc1)

        # Dropout：训练时随机丢弃30%神经元，防止过拟合
        if training:
            h_fc1 = tf.nn.dropout(h_fc1, rate=0.3)

        # 全连接层2（输出logits，不加softmax，loss函数内部会处理）
        logits = tf.matmul(h_fc1, self.W_fc2) + self.b_fc2
        return logits


model = myCNNModel()
optimizer = optimizers.Adam(learning_rate=1e-4)
print('模型建立完成！')

模型建立完成！


#### 计算 loss

In [10]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x, training=True)   # training=True 开启Dropout
        loss = compute_loss(logits, y)

    trainable_vars = [
        model.W_conv1, model.b_conv1,
        model.W_conv2, model.b_conv2,
        model.W_fc1,   model.b_fc1,
        model.W_fc2,   model.b_fc2
    ]
    grads = tape.gradient(loss, trainable_vars)
    optimizer.apply_gradients(zip(grads, trainable_vars))

    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x, training=False)      # training=False 关闭Dropout
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

#### 实际训练

In [11]:
train_data, test_data = mnist_dataset()

batch_size = 100   # 每次用100张图片训练

for epoch in range(3):   
    indices = np.random.permutation(train_data[0].shape[0])

    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start + batch_size]
        x_batch = tf.constant(train_data[0][batch_idx], dtype=tf.float32)
        y_batch = tf.constant(train_data[1][batch_idx], dtype=tf.int64)
        loss, accuracy = train_one_step(model, optimizer, x_batch, y_batch)

    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())

# 用测试集评估最终效果
loss, accuracy = test(model,
                      tf.constant(test_data[0], dtype=tf.float32),
                      tf.constant(test_data[1], dtype=tf.int64))
print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 0.045029152 ; accuracy 0.99
epoch 1 : loss 0.08335748 ; accuracy 0.98
epoch 2 : loss 0.10670183 ; accuracy 0.96
test loss 0.053358458 ; accuracy 0.9834
